# Entendimento dos dados

In [127]:
! pip install pandas pyarrow matplotlib huggingface_hub


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [128]:
import os
import time
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

from huggingface_hub import hf_hub_download

from huggingface_hub import HfFileSystem

In [129]:
HF_REPO = "hao-li/AIDev"
HF_REVISION = (
    "v5"  # pin a release (the tags are listed on the dataset card); "main" = latest
)

# Tables are downloaded once into the Hugging Face cache (~/.cache/huggingface), so
# re-running the notebook does not download again. If you already have the parquet
# files (e.g. a clone of the dataset repo), point AIDEV_DATA_DIR at that folder.
DATA_DIR = os.environ.get("AIDEV_DATA_DIR")


def table_path(name: str) -> Path:
    """Local path of `.parquet`, downloading it on first use."""
    if DATA_DIR:
        return Path(DATA_DIR) / f"{name}.parquet"
    return Path(
        hf_hub_download(
            HF_REPO, f"{name}.parquet", repo_type="dataset", revision=HF_REVISION
        )
    )


OUT_DIR = Path(
    "../dados/em_processamento"
)  # tables and figures written by this notebook
OUT_DIR.mkdir(exist_ok=True)
SCOPE = "pop"  # "all" ou "pop" = AIDev-pop (repositories with > 100 stars, 362 MB)
# "all" = every repository (all 7.6M PRs, a one-off 6.7 GB download)
PR_TABLE = "all_pull_request" if SCOPE == "all" else "pull_request"
REPO_TABLE = "all_repository" if SCOPE == "all" else "repository"

# Column sets reused throughout. Note what is NOT here: title, body, patch.
LIGHT_PR_COLS = [
    "id",
    "number",
    "agent",
    "user",
    "state",
    "created_at",
    "closed_at",
    "merged_at",
    "repo_url",
]
BATCH_ROWS = 65_536  # rows per streamed batch; smaller = lower peak memory, same speed

ALL_TABLES = [
    # full scope (every repository)
    "all_pull_request",
    "all_repository",
    "all_user",
    # AIDev-pop (> 100 stars) and its per-PR sub-resources
    "pull_request",
    "repository",
    "pr_comments",
    "pr_reviews",
    "pr_review_comments",
    "pr_commits",
    "pr_commit_details",
    "pr_timeline",
    "issue",
    "related_issue",
]

In [130]:
fs = HfFileSystem()
rows = []
for name in ALL_TABLES:
    with fs.open(f"datasets/{HF_REPO}@{HF_REVISION}/{name}.parquet", "rb") as f:
        md = pq.ParquetFile(f).metadata  # footer only, nothing else is read
    size_by_col: dict[str, int] = {}
    for rg in range(md.num_row_groups):
        for ci in range(md.num_columns):
            col = md.row_group(rg).column(ci)
            size_by_col[col.path_in_schema] = (
                size_by_col.get(col.path_in_schema, 0) + col.total_compressed_size
            )
    total = sum(size_by_col.values())
    heavy = sorted(size_by_col.items(), key=lambda kv: -kv[1])[:2]
    rows.append(
        {
            "table": name,
            "rows": f"{md.num_rows:,}",
            "GB on disk": f"{total / 1e9:.2f}",
            "heaviest columns": ", ".join(f"{c} {s / total:.0%}" for c, s in heavy),
        }
    )
pd.DataFrame(rows).set_index("table")

,rows,GB on disk,heaviest columns
table,,,
all_pull_request,"7,685,281",6.73,"body 87%, title 4%"
all_repository,"957,209",0.06,"url 42%, full_name 41%"
all_user,"399,962",0.01,"login 38%, created_at 33%"
pull_request,"361,296",0.36,"body 90%, title 4%"
repository,"25,402",0.00,"url 38%, full_name 37%"
pr_comments,"676,098",0.69,"body 98%, created_at 1%"
pr_reviews,"541,831",0.21,"body 93%, submitted_at 2%"
pr_review_comments,"291,248",0.33,"diff_hunk 66%, body 24%"
pr_commits,"1,718,829",0.34,"message 77%, sha 20%"


In [131]:
repos = pd.DataFrame(columns=['x'])
users = pd.DataFrame(columns=['x'])
pop_repos = pd.DataFrame(columns=['x'])

# repos = pd.read_parquet(f"hf://datasets/{HF_REPO}@{HF_REVISION}/all_repository.parquet")
users = pd.read_parquet(f"hf://datasets/{HF_REPO}@{HF_REVISION}/all_user.parquet")
pop_repos = pd.read_parquet(f"hf://datasets/{HF_REPO}@{HF_REVISION}/repository.parquet")
# print(
#     f"all_repository: {len(repos):,} rows | all_user: {len(users):,} rows | "
#     f"repository (>100 stars): {len(pop_repos):,} rows"
# )
# repos[["full_name", "language", "stars", "forks", "license"]].head()

In [132]:
repos.head()

,x


In [133]:
repos.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   x       0 non-null      object
dtypes: object(1)
memory usage: 132.0+ bytes


In [134]:
repos.describe()

,x
count,0
unique,0
top,NaN
freq,NaN


In [135]:
users.head()

,id,login,followers,following,created_at
0,238120892.0,vilelaassessoriasc-hue,0.0,0.0,2025-10-15T01:04:18Z
1,117571344.0,dylanngn,0.0,1.0,2022-11-06T10:44:21Z
2,12562703.0,teweldemat,0.0,0.0,2015-05-22T16:14:35Z
3,84263340.0,Adell02,2.0,4.0,2021-05-16T06:48:33Z
4,9817220.0,Rnijland,0.0,0.0,2014-11-18T06:57:36Z


In [136]:
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 399962 entries, 0 to 399961
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          399658 non-null  float64
 1   login       399962 non-null  str    
 2   followers   399658 non-null  float64
 3   following   399658 non-null  float64
 4   created_at  399658 non-null  str    
dtypes: float64(3), str(2)
memory usage: 26.9 MB


In [137]:
users.describe()

,id,followers,following
count,3.996580e+05,399658.000000,399658.000000
mean,1.055884e+08,22.039101,12.781040
std,8.512648e+07,434.453044,564.157121
min,7.000000e+00,0.000000,0.000000
25%,2.528131e+07,0.000000,0.000000
50%,8.868957e+07,1.000000,1.000000
75%,1.832830e+08,5.000000,5.000000
max,2.744340e+08,164726.000000,301606.000000


In [138]:
pop_repos.head()

,id,url,license,full_name,is_forked,language,forks,stars
0,922873328,https://api.github.com/repos/willmiao/ComfyUI-...,GPL-3.0,willmiao/ComfyUI-Lora-Manager,False,Python,101,1065
1,373383253,https://api.github.com/repos/hoc081098/FlowExt,MIT,hoc081098/FlowExt,False,Kotlin,28,416
2,651622038,https://api.github.com/repos/allenai/open-inst...,Apache-2.0,allenai/open-instruct,False,Python,531,3684
3,42267778,https://api.github.com/repos/NuGetPackageExplo...,MIT,NuGetPackageExplorer/NuGetPackageExplorer,False,C#,452,2537
4,5697090,https://api.github.com/repos/wookayin/dotfiles,MIT,wookayin/dotfiles,False,Lua,112,400


In [139]:
pop_repos.info()

<class 'pandas.DataFrame'>
RangeIndex: 25402 entries, 0 to 25401
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         25402 non-null  int64
 1   url        25402 non-null  str  
 2   license    23189 non-null  str  
 3   full_name  25402 non-null  str  
 4   is_forked  25402 non-null  bool 
 5   language   24749 non-null  str  
 6   forks      25402 non-null  int64
 7   stars      25402 non-null  int64
dtypes: bool(1), int64(3), str(4)
memory usage: 3.4 MB


In [140]:
pop_repos.describe()

,id,forks,stars
count,2.540200e+04,25402.000000,25402.00000
mean,5.026672e+08,611.086253,3511.96394
std,3.989228e+08,3192.652506,12475.58927
min,1.300000e+02,0.000000,101.00000
25%,1.228054e+08,33.000000,200.00000
50%,4.249520e+08,92.000000,502.00000
75%,8.961401e+08,307.000000,1921.75000
max,1.197330e+09,242815.000000,453366.00000


In [141]:
t0 = time.time()
prs = pd.read_parquet(
    table_path(PR_TABLE), columns=LIGHT_PR_COLS, dtype_backend="pyarrow"
)
mb = prs.memory_usage(deep=True).sum() / 1e6

print(
    f"{len(prs):,} PRs x {len(LIGHT_PR_COLS)} columns = {mb:,.0f} MB in RAM "
    f"({time.time() - t0:.1f} s incl. download on first run)"
)
prs.groupby("agent").size().sort_values(ascending=False).rename("# PR").to_frame()

361,296 PRs x 9 columns = 61 MB in RAM (0.2 s incl. download on first run)


,# PR
agent,
Claude_Code,172597
Copilot,98384
OpenAI_Codex,62731
Cursor,12908
Devin,11219
Google_Jules,3457


# Top Devs

In [142]:
df_prs = prs

In [143]:
df_prs

,id,number,agent,user,state,created_at,closed_at,merged_at,repo_url
0,3256881529,3690,OpenAI_Codex,MontrealAI,closed,2025-07-23T15:44:16Z,2025-07-23T15:44:29Z,2025-07-23T15:44:29Z,https://api.github.com/repos/MontrealAI/AGI-Al...
1,3122476985,24,OpenAI_Codex,puncsky,open,2025-06-05T19:57:59Z,<NA>,<NA>,https://api.github.com/repos/cuckoo-network/cu...
2,3122488612,62,OpenAI_Codex,charlesvestal,closed,2025-06-05T20:01:46Z,2025-06-05T20:03:40Z,2025-06-05T20:03:40Z,https://api.github.com/repos/charlesvestal/ext...
3,3087726186,666,OpenAI_Codex,MontrealAI,closed,2025-05-23T23:39:13Z,2025-05-23T23:39:23Z,2025-05-23T23:39:23Z,https://api.github.com/repos/MontrealAI/AGI-Al...
4,4125636513,18887,OpenAI_Codex,Ufodar,open,2026-03-24T06:41:04Z,<NA>,<NA>,https://api.github.com/repos/anomalyco/opencode
...,...,...,...,...,...,...,...,...,...
361291,3536159348,39803,Copilot,Copilot,closed,2025-10-21T11:41:48Z,2025-10-29T06:37:22Z,<NA>,https://api.github.com/repos/camunda/camunda
361292,4104386263,2802,Copilot,Copilot,closed,2026-03-19T21:48:25Z,2026-03-23T22:59:46Z,2026-03-23T22:59:46Z,https://api.github.com/repos/kubernetes-client...
361293,4104395262,21859,Copilot,Copilot,closed,2026-03-19T21:51:00Z,2026-03-20T06:03:01Z,<NA>,https://api.github.com/repos/github/gh-aw
361294,4104398985,9047,Copilot,Copilot,closed,2026-03-19T21:52:10Z,2026-03-19T21:52:29Z,2026-03-19T21:52:29Z,https://api.github.com/repos/Z3Prover/z3


# Filtrando somente PRs cujo merge foi realizado

In [144]:
df_prs = df_prs[~df_prs['merged_at'].isna()]

# Eliminando PRs de bots

In [145]:
df_prs = df_prs[~df_prs['user'].str.contains(r'\[bot\]')]

In [152]:
df_prs

,id,number,agent,user,state,created_at,closed_at,merged_at,repo_url
0,3256881529,3690,OpenAI_Codex,MontrealAI,closed,2025-07-23T15:44:16Z,2025-07-23T15:44:29Z,2025-07-23T15:44:29Z,https://api.github.com/repos/MontrealAI/AGI-Al...
2,3122488612,62,OpenAI_Codex,charlesvestal,closed,2025-06-05T20:01:46Z,2025-06-05T20:03:40Z,2025-06-05T20:03:40Z,https://api.github.com/repos/charlesvestal/ext...
3,3087726186,666,OpenAI_Codex,MontrealAI,closed,2025-05-23T23:39:13Z,2025-05-23T23:39:23Z,2025-05-23T23:39:23Z,https://api.github.com/repos/MontrealAI/AGI-Al...
7,3562923837,9,OpenAI_Codex,Jackhammer9,closed,2025-10-28T18:46:29Z,2025-10-28T19:26:46Z,2025-10-28T19:26:46Z,https://api.github.com/repos/Jackhammer9/Reduino
9,3713692276,4219,OpenAI_Codex,shai-almog,closed,2025-12-10T05:45:47Z,2025-12-10T18:07:24Z,2025-12-10T18:07:24Z,https://api.github.com/repos/codenameone/Coden...
...,...,...,...,...,...,...,...,...,...
361286,3536090828,2952,Copilot,Copilot,closed,2025-10-21T11:22:10Z,2025-10-22T14:51:34Z,2025-10-22T14:51:34Z,https://api.github.com/repos/open-telemetry/se...
361288,3536117776,3393,Copilot,Copilot,closed,2025-10-21T11:30:08Z,2025-10-30T16:53:18Z,2025-10-30T16:53:18Z,https://api.github.com/repos/obophenotype/cell...
361290,3536139323,2083,Copilot,Copilot,closed,2025-10-21T11:36:03Z,2025-10-21T12:59:22Z,2025-10-21T12:59:22Z,https://api.github.com/repos/github/gh-aw
361292,4104386263,2802,Copilot,Copilot,closed,2026-03-19T21:48:25Z,2026-03-23T22:59:46Z,2026-03-23T22:59:46Z,https://api.github.com/repos/kubernetes-client...


In [146]:
grupos = df_prs.groupby('user')

In [147]:
top_devs = grupos['id'].count().sort_values(ascending=False).head(100)

In [148]:
top_devs

user
Copilot         59643
tamnd           11905
MontrealAI       3795
Yeraze           1321
lancy            1306
                ...  
mikasoukhov       209
delano            203
AtlantisPleb      199
JAORMX            199
rudrankriyam      199
Name: id, Length: 100, dtype: int64[pyarrow]

# Usuários

In [149]:
users

,id,login,followers,following,created_at
0,238120892.0,vilelaassessoriasc-hue,0.0,0.0,2025-10-15T01:04:18Z
1,117571344.0,dylanngn,0.0,1.0,2022-11-06T10:44:21Z
2,12562703.0,teweldemat,0.0,0.0,2015-05-22T16:14:35Z
3,84263340.0,Adell02,2.0,4.0,2021-05-16T06:48:33Z
4,9817220.0,Rnijland,0.0,0.0,2014-11-18T06:57:36Z
...,...,...,...,...,...
399957,75183449.0,Bamboo92,0.0,0.0,2020-11-28T20:35:33Z
399958,61661885.0,Pletor,1.0,0.0,2020-03-01T17:20:43Z
399959,97606492.0,erikaromppanen,1.0,1.0,2022-01-12T12:53:16Z
399960,1041066.0,luckyakhi,1.0,0.0,2011-09-10T17:52:48Z


In [150]:
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 399962 entries, 0 to 399961
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          399658 non-null  float64
 1   login       399962 non-null  str    
 2   followers   399658 non-null  float64
 3   following   399658 non-null  float64
 4   created_at  399658 non-null  str    
dtypes: float64(3), str(2)
memory usage: 26.9 MB


In [151]:
users.describe()

,id,followers,following
count,3.996580e+05,399658.000000,399658.000000
mean,1.055884e+08,22.039101,12.781040
std,8.512648e+07,434.453044,564.157121
min,7.000000e+00,0.000000,0.000000
25%,2.528131e+07,0.000000,0.000000
50%,8.868957e+07,1.000000,1.000000
75%,1.832830e+08,5.000000,5.000000
max,2.744340e+08,164726.000000,301606.000000


# Fontes consultadas

* Repositório do dataset
    - https://github.com/SAILResearch/AI_Teammates_in_SE3
* Understanding Parquet Files and the PyArrow Library
    - https://medium.com/@rangabashyam22/understanding-parquet-files-and-the-pyarrow-library-2bf65ffe4847
* Bot Details
    - https://github.com/marketplace/actions/bot-details
